# GitHub OSS Commit Dataset for Survival Analysis

## Overview

This notebook demonstrates the data processing pipeline for creating a standardized GitHub commit dataset suitable for Open Source Software (OSS) survival analysis and founder departure prediction.

### What this notebook does:
1. Loads GitHub commit data from repositories
2. Identifies founders (earliest committers) for each repository
3. Transforms raw commit data into a standardized schema
4. Creates examples with features: repo_id, author_login, is_founder, file_count, commit_sequence_num, author_total_commits, repo_total_commits, commit_timestamp
5. Outputs labels: 'founder' or 'contributor'

### Dataset Details:
- **Source**: HuggingFace dataset (AdhyanshVerma/open-github-major-repos)
- **Original Size**: 2.85M commit records from 98 repositories
- **Processed**: 500,000 examples from 13 repositories
- **Task**: Binary classification (founder vs contributor)

### Research Question:
What determines whether an open-source project survives its founder stepping away?

## Install Dependencies

This cell installs required packages. Packages pre-installed on Google Colab are skipped on Colab but installed locally to match Colab's environment.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# No additional packages needed - using standard library (json, pathlib, collections)
# If using this notebook with extended functionality, add packages here

print("Dependencies ready (using standard library only)")

## Imports

Import all required modules. The original script uses only standard library modules.

In [ ]:
from pathlib import Path
import json
from collections import defaultdict

print("Imports complete")

## Data Loading Helper

Loads the demo dataset from GitHub (for Colab) with local fallback. The dataset contains GitHub commit records with founder/contributor labels.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-a68c06-knowledge-redundancy-predicts-oss/main/round-1/dataset-1/demo/mini_demo_data.json"
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

print("Data loading helper defined")

## Load Data

Load the mini demo dataset containing GitHub commit examples.

In [ ]:
# Load the data
data = load_data()

# Extract examples from the dataset format
if "datasets" in data:
    examples = data["datasets"][0]["examples"]
    print(f"Loaded {len(examples)} examples from dataset")
else:
    examples = data
    print(f"Loaded {len(examples)} examples (raw format)")

# Display first example
if examples:
    print("\nFirst example:")
    print(f"  Output: {examples[0]['output']}")
    print(f"  Metadata: repo_id={examples[0]['metadata_repo_id']}, author={examples[0]['metadata_author']}")

## Configuration

Define processing parameters. For this demo, we use minimal settings.

In [ ]:
# Configuration parameters
MAX_EXAMPLES = len(examples)  # Use all examples from demo data
VERBOSE = True  # Print progress messages

print(f"Config: MAX_EXAMPLES={MAX_EXAMPLES}, VERBOSE={VERBOSE}")

## Process Data: Parse Input Features

Parse the input JSON strings to extract structured features from each example.

In [ ]:
# Parse input features from JSON strings
parsed_examples = []

for i, example in enumerate(examples[:MAX_EXAMPLES]):
    try:
        # Parse the input JSON string
        input_features = json.loads(example["input"])
        
        # Add output label
        input_features["output"] = example["output"]
        input_features["metadata"] = {
            "repo_id": example["metadata_repo_id"],
            "author": example["metadata_author"],
            "is_founder": example["metadata_is_founder"]
        }
        
        parsed_examples.append(input_features)
        
        if VERBOSE and (i + 1) % 5 == 0:
            print(f"Parsed {i + 1}/{len(examples[:MAX_EXAMPLES])} examples")
    except Exception as e:
        print(f"Error parsing example {i}: {e}")

print(f"\nSuccessfully parsed {len(parsed_examples)} examples")

# Show sample parsed example
if parsed_examples:
    print("\nSample parsed example:")
    sample = parsed_examples[0]
    for key, value in sample.items():
        if key != "metadata":
            print(f"  {key}: {value}")

## Analyze Repository Statistics

Group data by repository and compute statistics about founders vs contributors.

In [ ]:
# Group examples by repository
repos = defaultdict(list)
for example in parsed_examples:
    repo_id = example["repo_id"]
    repos[repo_id].append(example)

print(f"Found {len(repos)} repositories")
print("\nRepository statistics:")

repo_stats = []
for repo_id, commits in repos.items():
    # Count founders and contributors
    founders = [c for c in commits if c["is_founder"]]
    contributors = [c for c in commits if not c["is_founder"]]
    
    # Calculate average file counts
    avg_files_founder = sum(c["file_count"] for c in founders) / len(founders) if founders else 0
    avg_files_contributor = sum(c["file_count"] for c in contributors) / len(contributors) if contributors else 0
    
    stats = {
        "repo_id": repo_id,
        "total_commits": len(commits),
        "founder_commits": len(founders),
        "contributor_commits": len(contributors),
        "unique_contributors": len(set(c["author_login"] for c in commits if not c["is_founder"])),
        "avg_files_founder": avg_files_founder,
        "avg_files_contributor": avg_files_contributor
    }
    repo_stats.append(stats)
    
    print(f"\n  {repo_id}:")
    print(f"    Total commits: {stats['total_commits']}")
    print(f"    Founder commits: {stats['founder_commits']}")
    print(f"    Contributor commits: {stats['contributor_commits']}")
    print(f"    Unique contributors: {stats['unique_contributors']}")

## Results Visualization

Display key findings from the dataset analysis.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Prepare data for visualization
repo_names = [s["repo_id"].split("/")[-1] for s in repo_stats]
founder_commits = [s["founder_commits"] for s in repo_stats]
contributor_commits = [s["contributor_commits"] for s in repo_stats]

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('GitHub OSS Dataset Analysis', fontsize=14, fontweight='bold')

# Plot 1: Commits by type (founder vs contributor) for each repo
x = np.arange(len(repo_names))
width = 0.35

ax1 = axes[0, 0]
ax1.bar(x - width/2, founder_commits, width, label='Founder', color='#2E86AB')
ax1.bar(x + width/2, contributor_commits, width, label='Contributor', color='#A23B72')
ax1.set_xlabel('Repository')
ax1.set_ylabel('Number of Commits')
ax1.set_title('Commits by Type per Repository')
ax1.set_xticks(x)
ax1.set_xticklabels(repo_names, rotation=45, ha='right')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Plot 2: Average file count comparison
avg_files_founder = [s["avg_files_founder"] for s in repo_stats]
avg_files_contributor = [s["avg_files_contributor"] for s in repo_stats]

ax2 = axes[0, 1]
ax2.bar(x - width/2, avg_files_founder, width, label='Founder', color='#2E86AB')
ax2.bar(x + width/2, avg_files_contributor, width, label='Contributor', color='#A23B72')
ax2.set_xlabel('Repository')
ax2.set_ylabel('Average Files per Commit')
ax2.set_title('Average File Count by Author Type')
ax2.set_xticks(x)
ax2.set_xticklabels(repo_names, rotation=45, ha='right')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# Plot 3: Unique contributors per repo
unique_contributors = [s["unique_contributors"] for s in repo_stats]

ax3 = axes[1, 0]
ax3.bar(repo_names, unique_contributors, color='#F18F01')
ax3.set_xlabel('Repository')
ax3.set_ylabel('Number of Unique Contributors')
ax3.set_title('Unique Contributors per Repository')
ax3.set_xticks(x)
ax3.set_xticklabels(repo_names, rotation=45, ha='right')
ax3.grid(axis='y', alpha=0.3)

# Plot 4: Dataset summary pie chart
ax4 = axes[1, 1]
total_founders = sum(founder_commits)
total_contributors = sum(contributor_commits)
ax4.pie([total_founders, total_contributors], labels=['Founder', 'Contributor'], 
        autopct='%1.1f%%', colors=['#2E86AB', '#A23B72'], startangle=90)
ax4.set_title('Overall Dataset Composition')

plt.tight_layout()
plt.show()

# Print summary statistics
print("\n" + "="*60)
print("DATASET SUMMARY")
print("="*60)
print(f"Total examples processed: {len(parsed_examples)}")
print(f"Number of repositories: {len(repos)}")
print(f"Total founder commits: {total_founders}")
print(f"Total contributor commits: {total_contributors}")
print(f"\nAverage commits per repo: {len(parsed_examples) / len(repos):.1f}")
print(f"Average unique contributors per repo: {np.mean(unique_contributors):.1f}")
print("="*60)

## Export Processed Data

Save the processed dataset in the standardized format for use in downstream analysis.

In [ ]:
# Recreate the standardized output format
output_examples = []

for example in parsed_examples:
    # Reconstruct the standardized example format
    input_features = {
        "repo_id": example["repo_id"],
        "repo_name": example["repo_name"],
        "author_login": example["author_login"],
        "is_founder": example["is_founder"],
        "file_count": example["file_count"],
        "commit_sequence_num": example["commit_sequence_num"],
        "author_total_commits": example["author_total_commits"],
        "repo_total_commits": example["repo_total_commits"],
        "commit_timestamp": example["commit_timestamp"]
    }
    
    output_example = {
        "input": json.dumps(input_features),
        "output": example["output"],
        "metadata_repo_id": example["metadata"]["repo_id"],
        "metadata_author": example["metadata"]["author"],
        "metadata_is_founder": example["metadata"]["is_founder"],
        "metadata_task_type": "classification",
        "metadata_n_classes": 2
    }
    
    output_examples.append(output_example)

# Create output in the same format as the original
output = {
    "datasets": [
        {
            "dataset": "github_oss_commits",
            "examples": output_examples
        }
    ]
}

# Save to file (optional - for demo purposes)
output_path = Path("processed_demo_output.json")
with open(output_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved {len(output_examples)} processed examples to {output_path}")
print("\nDemo notebook completed successfully!")